In [1]:
%load_ext autoreload
%autoreload 2
    

In [2]:
# DN = 'C://work/dev/python/progs/texts/attack_classifier/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd

# train

В функции get_opt_thresh src/funcs.py добавил переменную fpr_max, которая управляет долей ложноположительных ответов, сейчас на 10% стоит (fpr_max = 0.1), так выдает очень много вариантов. Можно поменять прям в коде этудолю и перезапустить обучение путем выполнения ячейки с train.

In [5]:
from src.train import train

train()

Some weights of RobertaModel were not initialized from the model checkpoint at data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch num - 1
epoch num - 2
epoch num - 3
epoch num - 4
epoch num - 5
epoch num - 6
epoch num - 7
epoch num - 8
epoch num - 9


Some weights of RobertaModel were not initialized from the model checkpoint at data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch num - 1
epoch num - 2
epoch num - 3
epoch num - 4
epoch num - 5
epoch num - 6
epoch num - 7
epoch num - 8


# pred

## Новые примеры

In [6]:
DN = 'data/artifacts/check_gpt'
fns = [f'{DN}/{fn}' for fn in os.listdir(DN) if not '.ipynb_checkpoints' in fn]
fns

['data/artifacts/check_gpt/20250105_mp_weixin_qq_com_report_0xc2f7fa57_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250103_cyfirma_com_report_0xcee0958_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250103_malwarebytes_com_report_0xd205d57c_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250101_unit42_paloaltonetworks_com_report_0x379d578f_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250103_hunters_security_report_0x71828e59_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250103_picussecurity_com_report_0x20e7635f_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250105_secureblink_com_report_0xdfcb7be_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250106_securelist_com_report_0x7e1c00f8_chatgpt_ttp.json',
 'data/artifacts/check_gpt/20250105_cyfirma_com_report_0xd4a37a78_chatgpt_ttp.json']

In [7]:
check_df = pd.concat([pd.read_json(fn) for fn in fns], ignore_index=True)

In [24]:
check_df.query('reason.str.contains("executing command shells and more.")')

,mitre_id,mitre_name,reason,is_correct
39,T1059,Command and Scripting Interpreter,executing command shells and more.,True


In [8]:
from src.predict import predict

pred_df = predict(check_df['reason'].tolist())

In [9]:
check_df = check_df.assign(pred_bert=pred_df['pred_str_tech'].tolist())

In [14]:
with pd.option_context('display.max_colwidth', 200):
    display(check_df.iloc[43:50])

,mitre_id,mitre_name,reason,is_correct,pred_bert
43,T1573.001,Encrypted Channel: Symmetric Cryptography,"The SCHANNEL security package, which supports SSL and TLS encryption on Windows.",False,()
44,T1190,Exploit Public-Facing Application,breached via the infamous ProxyLogon vulnerability (CVE-2021-26855) in Exchange servers.,True,"(T1190,)"
45,T1203,Exploitation for Client Execution,"By embedding a link to a malicious template, attackers can execute code on a victim s system without the decoy document itself being flagged as malicious.",True,()
46,T1137.006,Office Application Startup: Template Injection,Remote Template Injection is a sophisticated attack vector leveraging Microsoft Word s template functionality to deliver malicious payloads.,True,"(T1221,)"
47,T1204.002,User Execution: Malicious File,"Typically, a malicious Office file is attached to an email designed to appear as legitimate communication.",True,"(T1566,)"
48,T1566.001,Phishing: Spearphishing Attachment,"This attack is especially effective in spear-phishing campaigns, where attackers exploit trust to secretly execute malicious macros.",True,"(T1204,)"
49,T1027,Obfuscated Files or Information,Researchers also observed the use of camouflage techniques to obfuscate URLs in malicious documents.,True,"(T1027,)"


# Загрузка сравнительной таблицы

In [7]:
res_df = pd.read_excel('data/artifacts/GPT_BERT_verdicts.xlsx', engine='openpyxl')

In [12]:
res_df = res_df.iloc[:50]

In [14]:
res_df['pred_bert'].map

'(T1204,)'

# True и совпало

In [22]:
check_df.columns

Index(['mitre_id', 'mitre_name', 'reason', 'is_correct', 'pred_bert'], dtype='object')

In [30]:
true_sel = (check_df['is_correct'] == True)
eq_sel = check_df[['mitre_id', 'pred_bert']].apply(lambda x: any([it in x['mitre_id'] for it in x['pred_bert']]), axis=1)

check_df[true_sel & eq_sel]

,mitre_id,mitre_name,reason,is_correct,pred_bert
4,T1562.001,Impair Defenses: Disable or Modify Tools,The below AntiScan method attempts to bypass W...,True,"(T1518, T1562)"
5,T1057,Process Discovery,Continuously monitors running processes to det...,True,"(T1057,)"
8,T1053.005,Scheduled Task/Job: Scheduled Task,These files are configured to run automaticall...,True,"(T1053,)"
31,T1566,Phishing,phishing emails with attachments disguised as ...,True,"(T1566,)"
34,T1574,Hijack Execution Flow,DLL search order hijacking and side-loading,True,"(T1574,)"
36,T1068,Exploitation for Privilege Escalation,vulnerabilities in hardware drivers to escalat...,True,"(T1068,)"
38,T1055,Process Injection,a novel service injector designed to inject th...,True,"(T1055, T1543)"
39,T1059,Command and Scripting Interpreter,executing command shells and more.,True,"(T1059,)"
44,T1190,Exploit Public-Facing Application,breached via the infamous ProxyLogon vulnerabi...,True,"(T1190,)"
49,T1027,Obfuscated Files or Information,Researchers also observed the use of camouflag...,True,"(T1027,)"


In [31]:
# проверил 1 и 46 - deepseek а стороне bert, может еще что-то
check_df[true_sel & ~eq_sel]

,mitre_id,mitre_name,reason,is_correct,pred_bert
1,T1059,Command and Scripting Interpreter,The sample starts obtaining subsequent payload...,True,"(T1105,)"
3,T1219,Remote Access Tools,NonEuclid Remote Access Trojan (RAT) is a type...,True,()
7,T1486,Data Encrypted for Impact,Utilizing AES encryption to lock various file ...,True,()
9,T1566,Phishing,Unwitting targets receive a direct message (DM...,True,()
10,T1204,User Execution,What the target will actually download and ins...,True,()
11,T1539,Steal Web Session Cookie,"stealing credentials stored in most browsers, ...",True,()
13,T1589,Gather Victim Identity Information,Some of the stolen information includes friend...,True,()
16,T1001,Data Obfuscation,Token smuggling strategies employ encoding alg...,True,"(T1027,)"
18,T1566,Phishing,The attack began with a phishing campaign targ...,True,()
20,T1539,Steal Web Session Cookie,"facilitated credential theft, including cookie...",True,()


# Делаем прогнозы на всем датасете (чтобы получить метрики по количеству случаев с той же границей)

In [5]:
import joblib
from ruamel.yaml import YAML

import numpy as np

conf = YAML().load(open('params.yaml'))

mlb = joblib.load(conf['prep_text']['mlb_fn'])
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])

thresh_tech_l = joblib.load(conf['train_fin']['thresh_ttp_fn'])
thresh_tak_l = joblib.load(conf['train_fin']['thresh_fn'])

In [6]:
from src.train import load_external_data, enc_classes

conf = YAML().load(open('params.yaml'))
df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=False)



In [7]:
from src.predict import predict

pred_main = predict(df['sentence'].to_numpy().tolist())
pred_main.head(2)

,sentence,target,proba_tak,pred_tak,proba_tech,pred_tech,pred_str_tech,pred_str_tak
0,Adversaries may inject malicious code into pro...,1,"[0.9159165024757385, 0.945948600769043, 0.0647...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[3.096574118899298e-06, 0.0006365251028910279,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1055,)","(defense-evasion, privilege-escalation)"
1,"Before creating a window, graphical Windows-ba...",1,"[0.17777800559997559, 0.2613764703273773, 0.44...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[9.790064359549433e-05, 0.00014453123731072992...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1546, T1559)","(execution,)"


In [8]:
pred_main = pred_main.assign(ttp=df['ttp'])

## посчитаем эмбеддинги

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_ttp['feat_gen'] = conf_ttp['feat_gen_ttp'] 
emb_path = conf_ttp['feat_gen']['aug_emb_path']

smodel = SentenceTransformer(emb_path)

In [10]:
pred_main['enc_sents'] = smodel.encode(pred_main['sentence'].to_numpy()).tolist()

## находим границы на заданном уровне ошибок

In [11]:
border_l = []
for i in range(len(mlb_ttp.classes_)):
    perc_x = 0.05
    border_l.append(pred_main['proba_tech'].map(lambda x: x[i]).quantile(1-perc_x))

# Исследование плохих прогнозов

In [13]:
import pandas as pd
ex_num = 6
row = pred_df.iloc[ex_num]

row['pred_str_tech']

()

In [14]:
q_levels = []
for i, val in enumerate(row['proba_tech']):
    q_levels.append( ((pred_main['proba_tech'].map(lambda x: x[i]))>val).mean())

In [20]:
# датасет другой поэтому не все предсказывается

border_df = pd.DataFrame({'q_level':q_levels}).sort_values(by='q_level', ascending=True).assign(cls=lambda x: x.index.map(lambda i: mlb_ttp.classes_[i]))
border_df.head(3)

,q_level,cls
113,0.013519,T1484
13,0.014457,T1021
160,0.023933,T1569


In [21]:
# квантиль уровня похожести на T1059 
border_df.query('cls=="T1059"')

,q_level,cls
37,0.092788,T1059


In [48]:
border_df.query('cls=="T1570"')

,q_level,cls
161,0.047251,T1570


In [52]:
border_df.index.get_loc(161)

9

## близость к целевому классу

In [38]:
ttp = 'T1059'
idx_cls = np.where(mlb_ttp.classes_==ttp)[0][0]
idx_cls

37

In [24]:
# реальная метрика
proba = row['proba_tech'][idx_cls]
proba

0.07033061236143112

In [25]:
# текущая граница
thresh_tech_l[idx_cls]


0.345

In [28]:
(pred_main['proba_tech'].map(lambda x: x[idx_cls])>0.345).mean()

0.03728978007761966

In [26]:
# на заданном уровне граница 5% (квантиль)
border_l[idx_cls]

0.22371216416358936

## все близкие классы

### преодолевающие границу по всем классам

In [36]:
idx_close_cls = np.where(np.array(row['proba_tech'])>np.array(border_l))[0]
idx_close_cls

array([ 13,  22,  28,  41,  45,  47, 100, 107, 113, 139, 141, 160, 161])

In [37]:
mlb_ttp.classes_[idx_close_cls]

array(['T1021', 'T1037', 'T1047', 'T1068', 'T1072', 'T1078', 'T1210',
       'T1219', 'T1484', 'T1546', 'T1548', 'T1569', 'T1570'], dtype=object)

### просто близкие к классу

In [31]:
embeddings = smodel.encode([row['sentence']])
sims = cosine_similarity([embeddings[-1]], np.array(pred_main['enc_sents'].to_numpy().tolist()))[0]

In [55]:
thresh = 0.7
idx = pd.DataFrame({'sims':sims}).query('sims>@thresh').sort_values(by='sims', ascending=False).index
main_sim_df = pred_main[['sentence', 'ttp', 'pred_str_tech']].loc[idx]

In [56]:
main_sim_df.explode('ttp')['ttp'].value_counts()

ttp
T1570    1
T1548    1
Name: count, dtype: int64

In [46]:
with pd.option_context('display.max_colwidth', 500):
    display(main_sim_df)

,sentence,ttp,pred_str_tech
21707,They then successfully escalated to SYSTEM privileges via Cobalt Strike’s built-in “named pipe impersonation” (GetSystem) functionality.,[T1570],()
3218,APT37 has a function in the initial dropper to bypass Windows UAC in order to execute the next payload with higher privileges.,[T1548],"(T1548,)"
29068,Privilege escalation and lateral movement often depend on software utilities running from the command line.,[],()


In [47]:
row['sentence']

'administrative privileges were leveraged during the lateral movement to execute a PowerShell-based Cobalt Strike payload'

In [59]:
pred_main[pred_main.sentence.str.contains('PowerShell')].explode('ttp')['ttp'].value_counts()

ttp
T1059    211
T1027     54
T1105     53
T1140     27
T1047     18
        ... 
T1006      1
T1134      1
T1556      1
T1197      1
T1135      1
Name: count, Length: 75, dtype: int64

## сколько и какие пересекают грань

In [39]:
idx_cls

37

In [40]:
(pred_main['proba_tech'].map(lambda x: x[idx_cls])>proba).mean()

0.09278783958602846

In [64]:
import pandas as pd
sel_true = pred_main['ttp'].map(lambda x: ttp in x)
with pd.option_context('display.max_colwidth', 500):
    display(pred_main.loc[(pred_main['proba_tech'].map(lambda x: x[idx_cls])>proba), ['ttp', 'sentence']].head())

,ttp,sentence
182,[T1003.002],A number of tools can be used to retrieve the SAM file through in-memory techniques:
183,[T1003.002],"Alternatively, the SAM can be extracted from the Registry with Reg:"
184,[T1003.002],Creddump7 can then be used to process the SAM database locally to retrieve hashes.
252,[T1003.004],Reg can be used to extract from the Registry. Mimikatz can be used to extract secrets from memory.
382,[T1555.005],Adversaries may acquire user credentials from password managers by extracting the master password and/or plain-text credentials from memory. Adversaries may extract credentials from memory via [Exploitation for Credential Access].


## предсказания

In [36]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.049).mean()

NameError: name 'ttp_df' is not defined

## статистика по словам

In [ ]:
index = 31728

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[0:])

<div class='alert alert-info'>
В целом текст похож, не хватило cut-off 
</div>

## deepseek и подход изучаем

ниже дам предложение, классифицируй его как технику MITRE (https://attack.mitre.org/):
"access critical system resources, including the LSASS memory space"

In [75]:
pred_main.loc[pred_main.sentence.str.contains('similar'), 'ttp'].explode('ttp').value_counts()

ttp
T1027        11
T1140         7
T1036.005     7
T1574.002     4
T1057         4
             ..
T1606.001     1
T1608.005     1
T1562.010     1
T1218.012     1
T1560.003     1
Name: count, Length: 89, dtype: int64

# Качество

Тут с FP границей на 1%, из-за того метрика так просела

In [20]:
pred_main = pred_main.assign(target=df.target_ttp)


In [21]:

from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_main['target'].values.tolist()), 
                                                    np.array(pred_main['pred_tech'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_main['target'].values.tolist()), 
                                                    np.array(pred_main['pred_tech'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

(0.6820146066563898, 0.6145193913944036)

## может tokenizer не хорошо разделил

In [61]:
from transformers import RobertaTokenizer, RobertaModel

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':128, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


In [65]:
tok_d = tokenizer(row['sentence'], **tokenizer_opts)
tok_d

{'input_ids': tensor([[    0, 46119, 39850,  3693, 24073,    58, 26032,   148,     5, 30972,
          2079,     7, 11189,    10, 47734,    12,   805, 11776, 24044, 29239,
             2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [69]:
tokenizer.decode(tok_d['input_ids'][0])

'<s>administrative privileges were leveraged during the lateral movement to execute a PowerShell-based Cobalt Strike payload</s>'

In [75]:
tokenizer.decode([47734])

' PowerShell'